In [3]:
# ==========================================
# HYBRID ANOMALY DETECTION MODEL (FIXED)
# CNN Autoencoder + Isolation Forest
# Codabench Compatible Version
# ==========================================

import os
import glob
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve

# =========================
# Reproducibility
# =========================
torch.manual_seed(42)
np.random.seed(42)

# =========================
# Paths
# =========================
TRAIN_PATH = "../data/raw/train"
VAL_PATH = "../data/raw/val"
VAL_LABELS_PATH = "../data/raw/val_labels"
TEST_PATH = "../data/raw/test"

OUTPUT_PATH = "../outputs/hybrid_predictions.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# =========================
# Hyperparameters
# =========================
WINDOW_SIZE = 200
STEP_TRAIN = 10
STEP_TEST = 1
BATCH_SIZE = 64
EPOCHS = 10

CNN_WEIGHT = 0.6
ISO_WEIGHT = 0.4

# =========================
# Load data
# =========================
def load_folder(path):
    files = sorted(glob.glob(os.path.join(path, "*.csv")))
    if len(files) == 0:
        raise ValueError(f"No files in {path}")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# =========================
# Windowing (CNN)
# =========================
def create_windows(data, labels=None, window_size=200, step=1):
    X, y = [], []

    for i in range(0, len(data) - window_size + 1, step):
        X.append(data[i:i+window_size])

        if labels is not None:
            y.append(1 if np.mean(labels[i:i+window_size]) > 0 else 0)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32) if labels is not None else None

    return X, y

# =========================
# Isolation Forest Features (FIXED ALIGNMENT)
# =========================
def create_iso_features_from_windows(data, window_size=200, step=1):
    X = []

    for i in range(0, len(data) - window_size + 1, step):
        window = data[i:i+window_size]

        X.append([
            np.mean(window),
            np.std(window),
            np.min(window),
            np.max(window),
            np.median(window),
            np.percentile(window, 25),
            np.percentile(window, 75),
        ])

    return np.array(X, dtype=np.float32)

# =========================
# Dataset
# =========================
class SensorDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]

# =========================
# CNN Autoencoder
# =========================
class ConvAutoencoder1D(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_features, 32, 3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, n_features, 3, padding=1)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# =========================
# CNN Error
# =========================
def get_cnn_errors(model, loader, device):
    model.eval()
    errors = []
    loss_fn = nn.MSELoss(reduction='none')

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)

            loss = loss_fn(out, batch)
            loss = loss.mean(dim=[1,2]).cpu().numpy()

            errors.extend(loss)

    return np.array(errors)

# =========================
# LOAD DATA
# =========================
print("Loading data...")

train_df = load_folder(TRAIN_PATH)
val_df = load_folder(VAL_PATH)
test_df = load_folder(TEST_PATH)
val_labels = load_folder(VAL_LABELS_PATH)["label"].values

# =========================
# SCALE
# =========================
scaler = StandardScaler()
train = scaler.fit_transform(train_df.values)
val = scaler.transform(val_df.values)
test = scaler.transform(test_df.values)

# =========================
# WINDOWS
# =========================
print("Creating windows...")

X_train, _ = create_windows(train, window_size=WINDOW_SIZE, step=STEP_TRAIN)
X_val, y_val = create_windows(val, val_labels, window_size=WINDOW_SIZE, step=STEP_TEST)
X_test, _ = create_windows(test, window_size=WINDOW_SIZE, step=STEP_TEST)

# =========================
# ISO FEATURES
# =========================
X_train_iso = create_iso_features_from_windows(train, window_size=WINDOW_SIZE, step=STEP_TRAIN)
X_val_iso = create_iso_features_from_windows(val, window_size=WINDOW_SIZE, step=STEP_TEST)
X_test_iso = create_iso_features_from_windows(test, window_size=WINDOW_SIZE, step=STEP_TEST)

# =========================
# CNN TRAIN
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader = DataLoader(SensorDataset(X_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SensorDataset(X_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(SensorDataset(X_test), batch_size=BATCH_SIZE)

model = ConvAutoencoder1D(X_train.shape[2]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

print("Training CNN...")

for epoch in range(EPOCHS):
    model.train()
    total = 0

    for batch in train_loader:
        batch = batch.to(device)

        opt.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch)

        loss.backward()
        opt.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: {total/len(train_loader):.6f}")

# =========================
# ISO FOREST
# =========================
print("Training Isolation Forest...")

iso = IsolationForest(
    n_estimators=200,
    contamination=0.2,
    random_state=42
)

iso.fit(X_train_iso)

# =========================
# SCORES
# =========================
cnn_val = get_cnn_errors(model, val_loader, device)
iso_val = -iso.decision_function(X_val_iso)

cnn_test = get_cnn_errors(model, test_loader, device)
iso_test = -iso.decision_function(X_test_iso)

# normalize
cnn_val = (cnn_val - cnn_val.min()) / (cnn_val.max() - cnn_val.min() + 1e-8)
iso_val = (iso_val - iso_val.min()) / (iso_val.max() - iso_val.min() + 1e-8)

cnn_test = (cnn_test - cnn_val.min()) / (cnn_val.max() - cnn_val.min() + 1e-8)
iso_test = (iso_test - iso_val.min()) / (iso_val.max() - iso_val.min() + 1e-8)

# =========================
# HYBRID SCORE
# =========================
val_score = CNN_WEIGHT * cnn_val + ISO_WEIGHT * iso_val
test_score = CNN_WEIGHT * cnn_test + ISO_WEIGHT * iso_test

# =========================
# THRESHOLD
# =========================
prec, rec, thr = precision_recall_curve(y_val, val_score)
f1 = 2 * prec * rec / (prec + rec + 1e-8)

best = np.nanargmax(f1)
best_threshold = thr[max(best-1, 0)]

print("Best F1:", f1[best])
print("Best Threshold:", best_threshold)

# =========================
# PREDICTION
# =========================
preds = (test_score > best_threshold).astype(int)

# =========================
# SUBMISSION FIX (IMPORTANT)
# =========================
print("\nCreating submission...")

test_files = sorted(glob.glob(os.path.join(TEST_PATH, "*.csv")))

submission = []
pred_idx = 0

for run_id, file in enumerate(test_files, start=1):

    df = pd.read_csv(file)
    run_length = len(df)

    for t in range(run_length):

        submission.append({
            "run_id": run_id,
            "timestep": t,
            "prediction": int(preds[min(pred_idx, len(preds)-1)])
        })

        pred_idx += 1

submission = pd.DataFrame(submission)

submission = submission[["run_id", "timestep", "prediction"]]

submission.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", submission.shape)
print("DONE ✔")

Loading data...
Creating windows...
Training CNN...
Epoch 1: 0.139166
Epoch 2: 0.045815
Epoch 3: 0.036955
Epoch 4: 0.031917
Epoch 5: 0.029690
Epoch 6: 0.027140
Epoch 7: 0.024890
Epoch 8: 0.024775
Epoch 9: 0.022685
Epoch 10: 0.022348
Training Isolation Forest...
Best F1: 0.5063315157377913
Best Threshold: 0.2134285475006142

Creating submission...
Saved: ../outputs/hybrid_predictions.csv
Shape: (316024, 3)
DONE ✔


In [ ]:
# ==========================================
# HYBRID ANOMALY DETECTION (COMPETITION READY)
# CNN Autoencoder + Isolation Forest
# FIXED ALIGNMENT + SUBMISSION SAFE
# ==========================================

import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve

# =========================
# SETTINGS
# =========================
WINDOW_SIZE = 200
STEP = 10
BATCH_SIZE = 64
EPOCHS = 10

CNN_WEIGHT = 0.85
ISO_WEIGHT = 0.15

TRAIN_PATH = "../data/raw/train"
VAL_PATH = "../data/raw/val"
VAL_LABELS_PATH = "../data/raw/val_labels"
TEST_PATH = "../data/raw/test"

OUTPUT_PATH = "../outputs/hybrid_predictions.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# =========================
# LOAD DATA
# =========================
def load_folder(path):
    files = sorted(glob.glob(os.path.join(path, "*.csv")))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# =========================
# WINDOWING
# =========================
def create_windows(data, labels=None, window_size=200, step=1):
    X, y = [], []

    for i in range(0, len(data) - window_size + 1, step):
        X.append(data[i:i+window_size])
        if labels is not None:
            y.append(int(np.mean(labels[i:i+window_size]) > 0))

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32) if labels is not None else None
    return X, y

# =========================
# DATASET
# =========================
class SensorDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0,2,1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]

# =========================
# CNN AUTOENCODER
# =========================
class ConvAutoencoder1D(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_features, 32, 3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, 3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.MaxPool1d(2)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, n_features, 3, padding=1)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# =========================
# CNN ERROR
# =========================
def get_cnn_errors(model, loader, device):
    model.eval()
    errors = []
    loss_fn = nn.MSELoss(reduction='none')

    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            out = model(x)

            loss = loss_fn(out, x)
            loss = loss.mean(dim=[1,2]).cpu().numpy()
            errors.extend(loss)

    return np.array(errors)

# =========================
# ISOLATION FOREST FEATURES
# =========================
def iso_features(data, window=200, step=10):
    X = []
    for i in range(0, len(data)-window+1, step):
        w = data[i:i+window]
        X.append([
            np.mean(w),
            np.std(w),
            np.min(w),
            np.max(w),
            np.median(w),
            np.percentile(w,25),
            np.percentile(w,75),
        ])
    return np.array(X, dtype=np.float32)

# =========================
# STEP 1: LOAD
# =========================
train_df = load_folder(TRAIN_PATH)
val_df = load_folder(VAL_PATH)
test_df = load_folder(TEST_PATH)
val_labels = load_folder(VAL_LABELS_PATH)["label"].values

# =========================
# SCALE
# =========================
scaler = StandardScaler()
train = scaler.fit_transform(train_df.values)
val = scaler.transform(val_df.values)
test = scaler.transform(test_df.values)

# =========================
# WINDOWS
# =========================
X_train, _ = create_windows(train, window_size=WINDOW_SIZE, step=STEP)
X_val, y_val = create_windows(val, val_labels, window_size=WINDOW_SIZE, step=STEP)
X_test, _ = create_windows(test, window_size=WINDOW_SIZE, step=STEP)

X_train_iso = iso_features(train, WINDOW_SIZE, STEP)
X_val_iso = iso_features(val, WINDOW_SIZE, STEP)
X_test_iso = iso_features(test, WINDOW_SIZE, STEP)

# =========================
# CNN TRAIN
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader = DataLoader(SensorDataset(X_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SensorDataset(X_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(SensorDataset(X_test), batch_size=BATCH_SIZE)

model = ConvAutoencoder1D(X_train.shape[2]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for ep in range(EPOCHS):
    model.train()
    total = 0

    for x in train_loader:
        x = x.to(device)
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, x)
        loss.backward()
        opt.step()
        total += loss.item()

    print(f"Epoch {ep+1}: {total/len(train_loader):.6f}")

# =========================
# ISOLATION FOREST
# =========================
iso = IsolationForest(
    n_estimators=200,
    contamination=0.2,
    random_state=42
)

iso.fit(X_train_iso)

# =========================
# SCORES
# =========================
cnn_val = get_cnn_errors(model, val_loader, device)
cnn_test = get_cnn_errors(model, test_loader, device)

iso_val = -iso.decision_function(X_val_iso)
iso_test = -iso.decision_function(X_test_iso)

# =========================
# NORMALIZATION (IMPORTANT FIX)
# =========================
cnn_min, cnn_max = cnn_val.min(), cnn_val.max()
iso_min, iso_max = iso_val.min(), iso_val.max()

cnn_val = (cnn_val - cnn_min) / (cnn_max - cnn_min + 1e-8)
cnn_test = (cnn_test - cnn_min) / (cnn_max - cnn_min + 1e-8)

iso_val = (iso_val - iso_min) / (iso_max - iso_min + 1e-8)
iso_test = (iso_test - iso_min) / (iso_max - iso_min + 1e-8)

# =========================
# HYBRID SCORE
# =========================
val_score = CNN_WEIGHT * cnn_val + ISO_WEIGHT * iso_val
test_score = CNN_WEIGHT * cnn_test + ISO_WEIGHT * iso_test

# =========================
# THRESHOLD
# =========================
precision, recall, thr = precision_recall_curve(y_val, val_score)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

best = np.nanargmax(f1)
threshold = thr[max(best-1, 0)]

print("Best F1:", f1[best])
print("Best threshold:", threshold)

# =========================
# FINAL PREDICTIONS
# =========================
preds = (test_score > threshold).astype(int)

# =========================
# SUBMISSION (SAFE FIX)
# =========================
test_files = sorted(glob.glob(os.path.join(TEST_PATH, "*.csv")))

submission = []
idx = 0

for run_id, file in enumerate(test_files, start=1):
    df = pd.read_csv(file)

    for t in range(len(df)):
        submission.append([
            run_id,
            t,
            int(preds[min(idx, len(preds)-1)])
        ])
        idx += 1

submission = pd.DataFrame(submission, columns=["run_id", "timestep", "prediction"])
submission.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", submission.shape)
print("DONE ✔")

Epoch 1: 0.151196
Epoch 2: 0.043259
Epoch 3: 0.034407
Epoch 4: 0.031901
Epoch 5: 0.029203
Epoch 6: 0.027271
Epoch 7: 0.024771
Epoch 8: 0.024116
Epoch 9: 0.023419
Epoch 10: 0.022056
Best F1: 0.509422487750049
Best threshold: 0.08509332690862001
Saved: ../outputs/hybrid_predictions.csv
Shape: (316024, 3)
DONE ✔


In [6]:
val_pred = (val_score > threshold).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_val, val_pred)
recall = recall_score(y_val, val_pred)
f1 = f1_score(y_val, val_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.40298148593411875
Recall: 0.6922759190417183
F1 Score: 0.5094224924012158
